In [46]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [ ]:
def probable_role(df):

    roles = {}
    
    for col in df.columns:

        if (df[col] == 0).all() or df[col].isna().all():
            roles[col] = "Unknown or mixed type"
            continue

        if df[col].dtype == 'bool':
            roles[col] = "Boolean"
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            roles[col] = "Numeric"
            continue

        if pd.api.types.is_datetime64_any_dtype(df[col]):
            roles[col] = "Date-like"
            continue

        if len(df[col].unique())/len(df[col]) > 0.90:
            roles[col] = "Identifier-like"
            continue

        if len(df[col].unique()) > 2 & len(df[col].unique())/len(df[col]) < 0.05:
            roles[col] = "Categorical"
            continue

        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            roles[col] = "Free-text field"
            continue
            
        else:
            roles[col] = "Unknown or mixed type"
   
    return roles


In [86]:
def get_dataset_overview(df, dataset_name, roles):
    overview = {
        'filename': dataset_name,
        'rows': len(df),
        'columns': len(df.columns),
        'column_name': {},
        'technical_type': {},
        'non_missing_count': {},
        'missing_percent': {},
        'unique_values': {},
        'probable_role': {},
    }
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        overview['column_name'] = col
        overview['technical_type'][col] = str(df[col].dtype)
        overview['non_missing_count'][col] = len(df[col]) - missing_count
        overview['missing_percent'][col] = round(missing_count / len(df[col]), 4)
        overview['unique_values'][col] = int(df[col].nunique())
        overview['technical_type'][col] = roles[col]
    return overview

In [ ]:
def get_data_quality_profile(df):
    
    missing_threshold = 0.30
    sensitive_keywords = ['ssn', 'password', 'credit', 'phone', 'email',
                           'address', 'zip', 'dob', 'birth', 'salary', 'income', 'social_security']

    profile = {
        'Number of duplicate rows in each columns': {},
        'Columns containing only one distinct value': [],
        'Columns with a high percentage of missing values': [],
        'Columns that have mixed or inconsistent data type': [],
        'Columns that have identifier-like or extremely high-cardinality data type': [],
        'Columns that might contain sensitive data based on column name': []
    }

    for col in df.columns:
        profile['Number of duplicate rows in each columns'][col] = int(df.duplicated().sum())
        if df[col].nunique() == 1:
            profile['Columns containing only one distinct value'].append(col)
        if round(int(df[col].isna().sum()) / len(df[col]), 4) >= missing_threshold:
            profile['Columns with a high percentage of missing values'].append(col)
        if pd.api.types.infer_dtype(df[col]):
            profile['Columns that have mixed or inconsistent data type'].append(col)
        if len(df[col].unique())/len(df[col]) > 0.90:
            profile['Columns that have identifier-like or extremely high-cardinality data type'].append(col)
        if any(keyword in col.lower() for keyword in sensitive_keywords):
            profile['Columns that might contain sensitive data based on column name'].append(col)
            
    return profile

In [91]:
def descriptive_statistics(df):

    Categorical_Columns = [col for col in df.columns if (len(df[col].unique()) > 2) & (len(df[col].unique())/len(df[col]) < 0.05)]

    Categorical_Columns_Stats = {
        'Number of unique categories in the column': {},
        'Most frequent category or categories': {},
        'Frequency count of top 10 categories': {},
        'Frequency count percentage of top 10 categories': {},
        'Top categories when the column contains many unique values': {}
    }
    Numeric_Columns_Stats = {

    }
    Stats = [Categorical_Columns_Stats, Numeric_Columns_Stats]

    for col in Categorical_Columns:
        Categorical_Columns_Stats['Number of unique categories in the column'][col] = int(df[col].nunique())
        Categorical_Columns_Stats['Most frequent category or categories'][col] = df[col].value_counts().head(3)

        Categorical_Columns_Stats['Frequency count of top 10 categories'][col] = {}
        Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col] = {}

        for unique in df[col].value_counts().head(10):
            Categorical_Columns_Stats['Frequency count of top 10 categories'][col][unique] = int((df[col] == unique).sum())
            Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col][unique] = ((df[col] == unique).sum())/len(df[col])
        if df[col].nunique() > 10:
            Categorical_Columns_Stats['Top categories when the column contains many unique values'][col] = df[col].value_counts().head(3)

    return Categorical_Columns_Stats



In [92]:
csv_path = "../data/SupplyChainGHGEmissionFactors_v1.3.0_NAICS_CO2e_USD2022.csv"
df2 = pd.read_csv(csv_path)
descriptive_statistics(df2)

{'Number of unique categories in the column': {},
 'Most frequent category or categories': {},
 'Frequency count of top 10 categories': {},
 'Frequency count percentage of top 10 categories': {},
 'Top categories when the column contains many unique values': {}}

In [66]:
csv_path = "../data/SupplyChainGHGEmissionFactors_v1.3.0_NAICS_CO2e_USD2022.csv"
def generate_profile(csv_path):
    dataset_name = os.path.basename(csv_path)
    df2 = pd.read_csv(csv_path)
    probable_role_output = probable_role(df2)
    get_dataset_overview_output = get_dataset_overview(df2, dataset_name, probable_role_output)
    get_data_quality_profile_output = get_data_quality_profile(df2)

    return get_dataset_overview_output, get_data_quality_profile_output

generate_profile(csv_path)

({'filename': 'SupplyChainGHGEmissionFactors_v1.3.0_NAICS_CO2e_USD2022.csv',
  'rows': 1016,
  'columns': 8,
  'column_name': 'Reference USEEIO Code',
  'technical_type': {'2017 NAICS Code': 'Numeric',
   '2017 NAICS Title': 'Identifier-like',
   'GHG': 'Categorical',
   'Unit': 'Categorical',
   'Supply Chain Emission Factors without Margins': 'Numeric',
   'Margins of Supply Chain Emission Factors': 'Numeric',
   'Supply Chain Emission Factors with Margins': 'Numeric',
   'Reference USEEIO Code': 'Free-text field'},
  'non_missing_count': {'2017 NAICS Code': 1016,
   '2017 NAICS Title': 1016,
   'GHG': 1016,
   'Unit': 1016,
   'Supply Chain Emission Factors without Margins': 1016,
   'Margins of Supply Chain Emission Factors': 1016,
   'Supply Chain Emission Factors with Margins': 1016,
   'Reference USEEIO Code': 1016},
  'missing_percent': {'2017 NAICS Code': 0.0,
   '2017 NAICS Title': 0.0,
   'GHG': 0.0,
   'Unit': 0.0,
   'Supply Chain Emission Factors without Margins': 0.0,
  

In [ ]:
def get_descriptive_statistics(df, roles):
    """Calculates specific stats based on column attribute types."""
    stats = {}
    for col in df.columns:
        role = roles[col]['role']
        col_data = df[col]
        
        if role == 'Numeric':
            valid_data = col_data.dropna()
            if not valid_data.empty:
                q1 = valid_data.quantile(0.25)
                q3 = valid_data.quantile(0.75)
                iqr = q3 - q1
                outliers = valid_data[(valid_data < (q1 - 1.5 * iqr)) | (valid_data > (q3 + 1.5 * iqr))]
                
                stats[col] = {
                    'min': float(valid_data.min()),
                    'max': float(valid_data.max()),
                    'mean': float(valid_data.mean()),
                    'median': float(valid_data.median()),
                    'std': float(valid_data.std()),
                    'q1': float(q1),
                    'q3': float(q3),
                    'iqr': float(iqr),
                    'outlier_count': int(len(outliers))
                }
                
        elif role in ['Categorical/Text', 'Boolean']:
            val_counts = col_data.value_counts().head(10)
            stats[col] = {
                'top_categories': {str(k): int(v) for k, v in val_counts.items()}
            }
    return stats

In [ ]:
def get_relationships_between_variables(df, roles):
    """Calculates correlations between appropriate numeric columns."""
    relationships = {}
    numeric_cols = [col for col, info in roles.items() if info['role'] == 'Numeric']
    
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr()
        corr_unstacked = corr_matrix.abs().unstack()
        corr_unstacked = corr_unstacked[corr_unstacked < 1.0].dropna()
        
        if not corr_unstacked.empty:
            strongest = corr_unstacked.idxmax()
            relationships['strongest_pair'] = strongest
            relationships['correlation_value'] = round(corr_matrix.loc[strongest[0], strongest[1]], 4)
            relationships['skipped'] = False
    else:
        relationships['skipped'] = True
        relationships['reason'] = "Not enough numeric columns for correlation analysis."
        
    return relationships

In [ ]:
def generate_adaptive_visualizations(df, roles, plots_dir):
    """Selects and generates plots based on available attribute types."""
    generated_plots = []
    sns.set_theme(style="whitegrid")
    
    # 1. Missing Values Bar Chart
    missing_data = df.isna().sum()
    if missing_data.sum() > 0:
        plt.figure(figsize=(10, 6))
        missing_data[missing_data > 0].plot(kind='bar', color='salmon')
        plt.title('Missing Values per Column')
        plt.ylabel('Count of Missing Values')
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, 'plot_01_missing_values.png'))
        plt.close()
        generated_plots.append('plot_01_missing_values.png')

    numeric_cols = [col for col, info in roles.items() if info['role'] == 'Numeric']
    cat_cols = [col for col, info in roles.items() if info['role'] in ['Categorical/Text', 'Boolean']]
    plot_num = len(generated_plots) + 1

    # 2. Numeric Correlation Heatmap
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 8))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Numeric Correlation Heatmap')
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f'plot_{plot_num:02d}_correlation.png'))
        plt.close()
        generated_plots.append(f'plot_{plot_num:02d}_correlation.png')
        plot_num += 1

    # 3. Numeric Distributions (Histograms)
    for col in numeric_cols[:3]: 
        plt.figure(figsize=(10, 6))
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_dist_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1

    # 4. Categorical Frequencies (Bar Charts)
    for col in cat_cols[:3]:
        plt.figure(figsize=(10, 6))
        df[col].value_counts().head(10).plot(kind='bar', color='skyblue')
        plt.title(f'Top Categories in {col}')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_cat_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1

    return generated_plots

In [ ]:
def run_llm_insights(summary, output_dir, use_llm):
    """Passes structured summary to local LLM with graceful fallback."""
    prompt = f"""
    You are an expert data analyst. I have profiled a dataset and generated the following verified statistics in JSON format. 
    Using ONLY these facts, write 5 to 8 evidence-based insights. 
    
    You must include:
    - At least one data-quality finding.
    - At least one distribution finding.
    - At least one categorical finding.
    - At least one relationship finding (if applicable).
    - At least one limitation or potential-bias warning.
    - At least one recommended question for further investigation.

    Data Summary:
    {json.dumps(summary, indent=2)}

    Do not calculate new statistics. Do not invent facts. Write clearly and professionally.
    """
    
    with open(os.path.join(output_dir, 'llm_prompt.txt'), 'w', encoding='utf-8') as f: f.write(prompt)

    if not use_llm:
        msg = "LLM insights skipped (use_llm=False)."
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(msg)
        return msg

    try:
        response = requests.post('http://localhost:11434/api/generate', json={"model": "llama3", "prompt": prompt, "stream": False}, timeout=45)
        response.raise_for_status()
        insights = response.json().get('response', '')
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(insights)
        return insights
    except Exception as e:
        msg = f"Graceful Failure: LLM unavailable.\nError: {e}"
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(msg)
        return msg

In [ ]:
def build_markdown_report(dataset_name, summary, generated_plots, llm_insights, output_dir):
    """Compiles outputs into a final Markdown report."""
    md = f"# Data Profiling Report: {dataset_name}\n\n"
    md += f"## Dataset Overview\n- **Rows:** {summary['overview']['rows']:,}\n- **Columns:** {summary['overview']['columns']:,}\n\n"
    md += "## Data Quality Findings\n" + ("\n".join([f"- ⚠️ {w}" for w in summary['quality_warnings']]) if summary['quality_warnings'] else "- No major issues detected.") + "\n\n"
    md += f"## AI-Assisted Insights\n{llm_insights}\n\n"
    md += "## Visualizations\n" + "\n\n".join([f"![{plot}](plots/{plot})" for plot in generated_plots])
    
    with open(os.path.join(output_dir, 'report.md'), 'w', encoding='utf-8') as f: f.write(md)

In [ ]:
def generate_profile(csv_path, output_dir, use_llm=True):
    """Main Orchestrator Entry Point"""
    print(f"Starting analysis for {csv_path}...")
    dataset_name = os.path.basename(csv_path)
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    
    df = pd.read_csv(csv_path)
    roles = infer_column_roles(df)
    
    # Execute the modular analysis functions
    overview = get_dataset_overview(df, roles)
    quality = get_data_quality_profile(df, overview, roles)
    stats = get_descriptive_statistics(df, roles)
    relationships = get_relationships_between_variables(df, roles)
    
    # Combine into one master JSON for the LLM
    summary = {
        'overview': overview,
        'quality_warnings': quality,
        'descriptive_statistics': stats,
        'relationships': relationships
    }
    
    with open(os.path.join(output_dir, 'analysis_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=4)
        
    pd.DataFrame.from_dict(overview['column_info'], orient='index').to_csv(os.path.join(output_dir, 'column_profile.csv'), index_label='Column_Name')
    
    generated_plots = generate_adaptive_visualizations(df, roles, os.path.join(output_dir, 'plots'))
    llm_insights = run_llm_insights(summary, output_dir, use_llm)
    build_markdown_report(dataset_name, summary, generated_plots, llm_insights, output_dir)
    print(f"Success! Report generated at: {os.path.join(output_dir, 'report.md')}")